In [ ]:
from ase.io import read
from ase.visualize import view
import nqetools as nqe

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

tol_energy = 1.0e-3
tol_force = 1.0e-3
tol_position = 1.0e-3

n_beads = 4
timestep = 1.0  # fs
total_steps_md = 100
total_steps_plumed = 1_000

fix_com = True

stride = 10
temperature = 300
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'ase-mace'

# Expensive settings
driver_args = {'model': 'large',
               'device': 'cuda',
               'default_dtype': 'float64'}

# Cheap settings
driver_args = {'model': 'small',
               'device': 'cuda',
               'default_dtype': 'float32'}

# Plumed hills settings
n_bins = 100
stride_hills = 100
cv_limits = [None, None]

# # Plumed settings
plumed_type_opes = "opes-diff1"
plumed_args_opes = {'idx1': 1,
                    'idx2': 8,
                    'idx3': 0,
                    'barrier': 0.1,
                    'stride_hills': stride_hills}

# plumed_type_opes = "opes-dist"
# plumed_args_opes = {'idx1': 1,
#                     'idx2': 8,
#                     'barrier': 0.5,
#                     'stride_hills': stride_hills}

In [ ]:
# Make the system
atoms = read("malonaldehyde.traj")
atoms.center(vacuum=10.0)
# Delete the hydrogen atom
del atoms[-1]
del atoms[5]
atoms = nqe.add_hydrogen_at_distance(atoms, 0, 1, 1.0)
view(atoms)

In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          driver_args=driver_args)
atoms_opti, output_data_opti, output_desc_opti = output

# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          driver_args=driver_args,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position)
atoms_opti, output_data_opti, output_desc_opti = output
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
view(atoms_opti)

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    stride=1,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
view(atoms_md)

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
view(atoms_meta_md)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def load_plumed_colvar(path, field, derivative=False, x="time"):
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        header = f.readline().strip()

    prefix = "#! FIELDS"
    if not header.startswith(prefix):
        raise ValueError("First line must start with '#! FIELDS'.")

    # Extract column names after '#! FIELDS'
    names = header[len(prefix):].strip().split()
    if not names:
        raise ValueError("No column names found after '#! FIELDS'.")

    # Validate requested columns
    if x not in names:
        raise ValueError(f"x-axis column '{x}' not found. Available: {names}")
    if field not in names:
        raise ValueError(f"Field '{field}' not found. Available: {names}")

    data = np.loadtxt(path, comments="#")
    name_to_idx = {name: i for i, name in enumerate(names)}
    x_idx = name_to_idx[x]
    y_idx = name_to_idx[field]

    x_vals = data[:, x_idx]
    y_vals = data[:, y_idx]

    dt = x_vals[1] - x_vals[0]
    if derivative:
        if len(x_vals) < 2:
            raise ValueError("Not enough data points to compute derivative.")
        dy = np.gradient(y_vals, dt)
        y_vals = dy
    return x_vals, y_vals


def plot_field(path,
               field,
               *,
               x="time",
               title=None,
               show=True,
               save_to=None,
               derivative=False):
    x_vals, y_vals = load_plumed_colvar(path, field, derivative=derivative, x=x)

    fig, ax = plt.subplots()
    ax.plot(x_vals, y_vals)
    ax.set_xlabel(x)
    ax.set_ylabel(field)
    ax.set_title(title if title else f"{field} vs {x}")

    if save_to is not None:
        fig.savefig(save_to, bbox_inches="tight", dpi=150)
    if show:
        plt.show()

    return ax


plot_field("/home/louie/skunkworks/nqetools/examples/malonaldehyde/meta_md/COLVAR", "opes.rct", derivative=False)


In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_md)
nqe.plot_fes_series_1d(fes_arrays_meta_md, fes_times)

In [ ]:
# Run PIMD OPES metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
view(atoms_meta_pimd)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)

# Plot the free energy surface convergence
fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_pimd)
nqe.plot_fes_series_1d(fes_arrays_meta_pimd, fes_times)

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_series_1d_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])